In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_classic.agents import tool

In [3]:
@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location"""
    return "The weather in " + location + " is sunny"

In [4]:
get_weather.name

'get_weather'

In [5]:
get_weather.description

'Get the weather in a given location'

In [6]:
get_weather.args

{'location': {'title': 'Location', 'type': 'string'}}

In [7]:
from pydantic import BaseModel, Field
class SearchInput(BaseModel):
    query: str = Field(description="The search query")


In [8]:
@tool(args_schema=SearchInput)
def search_web(query: str) -> str:
    """Search the web for information"""
    return "The information for " + query + " is found"


In [9]:
search_web.args

{'query': {'description': 'The search query',
  'title': 'Query',
  'type': 'string'}}

In [10]:
search_web.run("What is the weather in Tokyo?")

'The information for What is the weather in Tokyo? is found'

In [11]:
from langchain_classic.chains.openai_functions.openapi import openapi_spec_to_openai_fn
from langchain_classic.utilities.openapi import OpenAPISpec

In [12]:
text = """
{
    "openapi": "3.0.0",
    "info": {
        "version": "1.0.0",
        "title": "Swagger Petstore",
        "license":{
            "name":"MIT"
        }
    },
    "servers": [
        {
            "url":"http://petstore.swagger.io/v1"
        }
    ],
    "paths": {
        "/pets": {
            "get": {
                "summary": "List all pets",
                "responses": {
                    "200": {
                        "description": "A list of pets",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "type": "array",
                                    "items": {
                                        "$ref": "#/components/schemas/Pet"
                                    }
                                }
                            }
                        }
                    }
                }
            }
        }
    },
    "components": {
        "schemas": {
            "Pet": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "name": {"type": "string"}
                }
            }
        }
    }
}


"""

In [13]:
spec = OpenAPISpec.from_text(text)

Attempting to load an OpenAPI 3.0.0 spec.  This may result in degraded performance. Convert your OpenAPI spec to 3.1.* spec for better support.


In [14]:
pet_openai_function, pet_callables = openapi_spec_to_openai_fn(spec)

In [15]:
pet_openai_function

[{'name': 'pets_get',
  'description': 'List all pets',
  'parameters': {'type': 'object', 'properties': {}}}]

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


In [17]:
model = ChatOpenAI(
    model="LongCat-Flash-Chat",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"  # 兼容 OpenAI 的第三方 API
).bind(
    functions=pet_openai_function
)




In [18]:
model.invoke("What are three pets names?")

AIMessage(content="Sure! Here are three popular pet names:\n\n1. **Bella** – A sweet and timeless name often given to dogs, cats, and even rabbits.  \n2. **Max** – A classic, energetic name commonly used for dogs and sometimes cats.  \n3. **Luna** – A celestial-inspired name that’s become very popular for pets of all kinds in recent years.\n\nLet me know if you'd like more suggestions or names for a specific type of pet! 🐾", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 17, 'total_tokens': 121, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'longcat-flash-chatai-api', 'system_fingerprint': None, 'id': '1d1222ff87a74036bced0c9cdd29c50d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3ed2-ec58-7730-ad60-2e90d5a4daa5-0', tool_calls=[], inva